In [1]:
import math

def precision_at_k(relevant, retrieved, k=1):
    retrieved = retrieved[:k]
    rel_set = set(relevant)
    return sum([1 for r in retrieved if r in rel_set]) / k


def recall_at_k(relevant, retrieved, k=10):
    rel_set = set(relevant)
    retrieved = retrieved[:k]
    return sum([1 for r in retrieved if r in rel_set]) / len(rel_set)


def mrr(relevant, retrieved):
    for idx, r in enumerate(retrieved, start=1):
        if r in relevant:
            return 1 / idx
    return 0


In [23]:
def f1_score(precision, recall):
    if precision + recall == 0:
        return 0
    return 2 * (precision * recall) / (precision + recall)

In [2]:
from elasticsearch import Elasticsearch
es = Elasticsearch("http://localhost:9200",
                   basic_auth = ('elasticsearch', 'e3TKzHmKRFWBP4gY--cjeQ'),
                   request_timeout=60,
                   )
es.ping() 
print(es.info())

{'name': 'LAPTOP-ANN0J427', 'cluster_name': 'elasticsearch', 'cluster_uuid': '6AxnonBCTU-8fGsKa_EYMQ', 'version': {'number': '9.1.3', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '0c781091a2f57de895a73a1391ff8426c0153c8d', 'build_date': '2025-08-24T22:05:04.526302670Z', 'build_snapshot': False, 'lucene_version': '10.2.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [3]:
# Lexical Search

def lexical_search(index_name, query_text, k=10):
    body = {
        "size": k,
        "query": {
            "multi_match": {
                "query": query_text,
                "fields": ["scheme_name", "description"]
            }
        }
    }
    res = es.search(index=index_name, body=body)
    return [int(hit["_source"]["scheme_id"]) for hit in res["hits"]["hits"]]


print("Lexical search function ready ✅")

Lexical search function ready ✅


### mahasbert

In [4]:
def semantic_search(index_name, query_text, model, k=10):
    query_vec = model.encode(query_text).tolist()

    body = {
        "size": k,
        "query": {
            "script_score": {
                "query": {"match_all": {}},
                "script": {
                    "source": "cosineSimilarity(params.qv, 'mahasbert_des_vector') + 1.0",
                    "params": {"qv": query_vec}
                }
            }
        }
    }

    
    res = es.search(index=index_name, body=body)
    return [int(hit["_source"]["scheme_id"]) for hit in res["hits"]["hits"]]


In [5]:
def hybrid_search(index_name, query_text, model, k=10, alpha=0.5):
    query_vec = model.encode(query_text).tolist()

    body = {
        "size": k,
        "query": {
            "bool": {
                "should": [
                    {
                        "multi_match": {
                            "query": query_text,
                            "fields": ["scheme_name^2", "description"]
                        }
                    },
                    {
                        "script_score": {
                            "query": {"match_all": {}},
                            "script": {
                                "source":
                                    "double bm25=_score; "
                                    "double vec=cosineSimilarity(params.qv, 'mahasbert_des_vector') + 1.0; "
                                    "return bm25*(1-params.alpha) + vec*params.alpha;",
                                "params": {
                                    "qv": query_vec,
                                    "alpha": alpha
                                }
                            }
                        }
                    }
                ]
            }
        }
    }

    
    res = es.search(index=index_name, body=body)
    return [int(hit['_source']['scheme_id']) for hit in res['hits']['hits']]


### indicsbert

In [14]:
def semantic_search(index_name, query_text, model, k=10):
    query_vec = model.encode(query_text).tolist()

    body = {
        "size": k,
        "query": {
            "script_score": {
                "query": {"match_all": {}},
                "script": {
                    "source": "cosineSimilarity(params.qv, 'indicsbert_des_vector') + 1.0",
                    "params": {"qv": query_vec}
                }
            }
        }
    }

    
    res = es.search(index=index_name, body=body)
    return [int(hit["_source"]["scheme_id"]) for hit in res["hits"]["hits"]]


In [15]:
def hybrid_search(index_name, query_text, model, k=10, alpha=0.5):
    query_vec = model.encode(query_text).tolist()

    body = {
        "size": k,
        "query": {
            "bool": {
                "should": [
                    {
                        "multi_match": {
                            "query": query_text,
                            "fields": ["scheme_name^2", "description"]
                        }
                    },
                    {
                        "script_score": {
                            "query": {"match_all": {}},
                            "script": {
                                "source":
                                    "double bm25=_score; "
                                    "double vec=cosineSimilarity(params.qv, 'indicsbert_des_vector') + 1.0; "
                                    "return bm25*(1-params.alpha) + vec*params.alpha;",
                                "params": {
                                    "qv": query_vec,
                                    "alpha": alpha
                                }
                            }
                        }
                    }
                ]
            }
        }
    }

    
    res = es.search(index=index_name, body=body)
    return [int(hit['_source']['scheme_id']) for hit in res['hits']['hits']]


In [10]:
import pandas as pd
test_english = pd.read_csv("../testSets/english.csv")
print("Test set loaded ✅")


Test set loaded ✅


In [16]:
import pandas as pd

def evaluate(model_name, model, test_df, index_name,query_col="query",relevant_col="relevant_scheme_ids") :
    
    results = []

    for _, row in test_df.iterrows():
        query = row[query_col]
        raw = row[relevant_col]

        # Remove brackets and spaces
        clean = raw.strip().replace("[", "").replace("]", "").replace(" ", "")

        # Handle empty case
        if clean == "":
            relevant = []
        else:
            relevant = list(map(int, clean.split(",")))

        # Run searches
        lex_res = lexical_search(index_name, query)
        sem_res = semantic_search(index_name, query, model)
        hyb_res = hybrid_search(index_name, query, model)

        # Collect results for all 3 search types
        for search_type, retrieved in [
            ("lexical", lex_res),
            ("semantic", sem_res),
            ("hybrid", hyb_res)
        ]:
            results.append({
                "model": model_name,
                "search_type": search_type,
                "query": query,
                "relevant_count": len(relevant),
                "precision": precision_at_k(relevant, retrieved),
                "recall": recall_at_k(relevant, retrieved),
                "mrr": mrr(relevant, retrieved)
            })

    return pd.DataFrame(results)


In [12]:
from loadModels import load_mahaSBERT

mahaSBERT_model = load_mahaSBERT()
print("mahaSBERT model loaded ✅")

df_maha = evaluate(
    model_name="mahaSBERT",
    model=mahaSBERT_model,
    test_df=test_english,
    index_name="english",
    query_col="query",
    relevant_col="relevant_scheme_ids",
)


mahaSBERT model loaded ✅


In [13]:
df_maha

,model,search_type,query,relevant_count,precision,recall,mrr
0,mahaSBERT,lexical,Cash amount scheme for farmers instead of food...,1,1.0,1.000000,1.000000
1,mahaSBERT,semantic,Cash amount scheme for farmers instead of food...,1,0.0,1.000000,0.333333
2,mahaSBERT,hybrid,Cash amount scheme for farmers instead of food...,1,1.0,1.000000,1.000000
3,mahaSBERT,lexical,Food scheme at discounted rates,1,1.0,1.000000,1.000000
4,mahaSBERT,semantic,Food scheme at discounted rates,1,1.0,1.000000,1.000000
...,...,...,...,...,...,...,...
214,mahaSBERT,semantic,Agricultural Marketing Board schemes,4,1.0,1.000000,1.000000
215,mahaSBERT,hybrid,Agricultural Marketing Board schemes,4,0.0,0.750000,0.500000
216,mahaSBERT,lexical,Schemes on Aaple Sarkar portal,56,1.0,0.071429,1.000000
217,mahaSBERT,semantic,Schemes on Aaple Sarkar portal,56,0.0,0.107143,0.500000


In [27]:
df_maha["f1"] = df_maha.apply(lambda row: f1_score(row["precision"], row["recall"]), axis=1)

In [17]:
from loadModels import load_indicSBERT
indicSBERT_model = load_indicSBERT()
print("indicSBERT model loaded ✅")

df_indic = evaluate(
    model_name="indicSBERT",
    model=indicSBERT_model,
    test_df=test_english,
    index_name="english",
    query_col="query",
    relevant_col="relevant_scheme_ids"
)


indicSBERT model loaded ✅


In [18]:
df_indic

,model,search_type,query,relevant_count,precision,recall,mrr
0,indicSBERT,lexical,Cash amount scheme for farmers instead of food...,1,1.0,1.000000,1.0
1,indicSBERT,semantic,Cash amount scheme for farmers instead of food...,1,1.0,1.000000,1.0
2,indicSBERT,hybrid,Cash amount scheme for farmers instead of food...,1,1.0,1.000000,1.0
3,indicSBERT,lexical,Food scheme at discounted rates,1,1.0,1.000000,1.0
4,indicSBERT,semantic,Food scheme at discounted rates,1,1.0,1.000000,1.0
...,...,...,...,...,...,...,...
214,indicSBERT,semantic,Agricultural Marketing Board schemes,4,1.0,0.750000,1.0
215,indicSBERT,hybrid,Agricultural Marketing Board schemes,4,0.0,1.000000,0.5
216,indicSBERT,lexical,Schemes on Aaple Sarkar portal,56,1.0,0.071429,1.0
217,indicSBERT,semantic,Schemes on Aaple Sarkar portal,56,0.0,0.125000,0.5


## final comparision

In [26]:
df_indic["f1"] = df_indic.apply(lambda row: f1_score(row["precision"], row["recall"]), axis=1)

In [28]:
combined = pd.concat([df_maha, df_indic], ignore_index=True)

In [29]:
summary = combined.groupby(["model", "search_type"]).agg({
    "precision": "mean",
    "recall": "mean",
    "mrr": "mean",
    "f1": "mean"
}).reset_index()


In [30]:
summary

,model,search_type,precision,recall,mrr,f1
0,indicSBERT,hybrid,0.794521,0.949837,0.871820,0.779224
1,indicSBERT,lexical,0.780822,0.946412,0.863584,0.765525
2,indicSBERT,semantic,0.575342,0.803767,0.662785,0.571863
3,mahaSBERT,hybrid,0.808219,0.946412,0.878669,0.792922
4,mahaSBERT,lexical,0.780822,0.946412,0.863584,0.765525
5,mahaSBERT,semantic,0.561644,0.775440,0.654680,0.561644


In [22]:
overall = combined.groupby("model").agg({
    "precision": "mean",
    "recall": "mean",
    "mrr": "mean"
})
print(overall)


            precision    recall       mrr
model                                    
indicSBERT   0.716895  0.900005  0.799397
mahaSBERT    0.716895  0.889422  0.798978


In [31]:
overall = combined.groupby("model").agg({
    "precision": "mean",
    "recall": "mean",
    "mrr": "mean",
    "f1": "mean"

})
print(overall)


            precision    recall       mrr        f1
model                                              
indicSBERT   0.716895  0.900005  0.799397  0.705537
mahaSBERT    0.716895  0.889422  0.798978  0.706697
